# EO-SAR Change Detection — Kaggle Training Notebook

| Property | Value |
|---|---|
| **Task** | Binary semantic segmentation — change detection |
| **Input** | Pre-event EO RGB + Post-event SAR grayscale → 4-channel tensor |
| **Model** | UNet++ with EfficientNet-B0 encoder |
| **Output** | Per-pixel mask: `0` = no change, `1` = change |
| **GPU** | Kaggle T4 (16 GB VRAM) |
| **Image size** | 512 × 512 (batch 8, mixed precision) |

---

This notebook is a **launcher only** — all implementation lives in the project modules (`datasets/`, `models/`, `losses/`, `utils/`).  
Cells execute `train.py` and `eval.py`; no model code lives here.

**Run order:** `1 → 2 → 3 → 4 → 5 → 6` for a full training run, then `7` for final test evaluation.

---
## § 1 — GPU Verification

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM            : {total:.1f} GB")
else:
    print("\n⚠  No GPU detected. Go to:  Run > Change runtime type > T4 GPU")

---
## § 2 — Install Dependencies

In [ ]:
%%capture install_log
!pip install -q \
    segmentation-models-pytorch==0.3.3 \
    timm==0.9.12 \
    albumentations==1.3.1 \
    tifffile==2024.2.12 \
    tensorboard==2.16.2

In [ ]:
import segmentation_models_pytorch as smp
import albumentations, tifffile, timm

print(f"segmentation-models-pytorch : {smp.__version__}")
print(f"albumentations              : {albumentations.__version__}")
print(f"tifffile                    : {tifffile.__version__}")
print(f"timm                        : {timm.__version__}")
print("All dependencies OK ✓")

---
## § 3 — Project Setup

Copies project code from Kaggle input to `/kaggle/working/Galaxy-AI` (writable),  
then patches `config.yaml` with the correct dataset path and T4-tuned hyperparameters.

> **Run this cell exactly once per session.** Re-running is safe (overwrites).

In [ ]:
import os, shutil, sys, yaml

# ── Paths ────────────────────────────────────────────────────────────────────
# Project source code (read-only Kaggle input)
SRC_CODE  = "/kaggle/input/datasets/gauravsingh101/galaxyeye/Galaxy Data/Galaxy AI"
# Dataset root  (read-only Kaggle input)
DATA_ROOT = "/kaggle/input/datasets/gauravsingh101/galaxyeye/Galaxy Data"
# Writable working directory
WORK_DIR  = "/kaggle/working/Galaxy-AI"

# ── Copy project code to writable workspace ──────────────────────────────────
for item in os.listdir(SRC_CODE):
    src = os.path.join(SRC_CODE, item)
    dst = os.path.join(WORK_DIR, item)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        os.makedirs(WORK_DIR, exist_ok=True)
        shutil.copy2(src, dst)

print(f"Project copied  : {WORK_DIR}")

# ── Switch to working dir so !python calls find modules ──────────────────────
os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print(f"Working dir     : {os.getcwd()}")

# ── Patch config.yaml for this Kaggle session ────────────────────────────────
CONFIG_PATH = os.path.join(WORK_DIR, "configs", "config.yaml")
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Dataset & output paths
cfg["data"]["root_dir"]          = DATA_ROOT
cfg["project"]["experiment_dir"] = os.path.join(WORK_DIR, "experiments")

# T4 GPU optimal settings  (512×512 / batch 8 / mixed precision)
cfg["data"]["image_size"]         = 512
cfg["data"]["num_workers"]        = 2
cfg["training"]["batch_size"]     = 8
cfg["training"]["mixed_precision"] = True
cfg["training"]["epochs"]         = 40
cfg["scheduler"]["T_max"]         = 40

with open(CONFIG_PATH, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"Config patched  : {CONFIG_PATH}")
print(f"  data.root_dir = {cfg['data']['root_dir']}")
print(f"  image_size    = {cfg['data']['image_size']}")
print(f"  batch_size    = {cfg['training']['batch_size']}")
print(f"  epochs        = {cfg['training']['epochs']}")
print(f"  mixed_prec    = {cfg['training']['mixed_precision']}")

---
## § 4 — Sanity Check: Dataset + Model

Verifies:
- All `train / val / test` directories exist
- One sample loads with correct shape and value range
- Model forward pass produces correct output shape

In [ ]:
import yaml, torch
from datasets.dataset import EOSARDataset, get_transforms
from models.model import build_model

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

# ── Directory check ───────────────────────────────────────────────────────────
print("Dataset directories:")
for split in ("train", "val", "test"):
    for sub in ("pre-event", "post-event", "target"):
        path   = os.path.join(cfg["data"]["root_dir"], split, sub)
        status = "✓" if os.path.isdir(path) else "✗  MISSING"
        count  = len(os.listdir(path)) if os.path.isdir(path) else 0
        print(f"  {status}  {split}/{sub}  ({count} files)")

# ── One sample ────────────────────────────────────────────────────────────────
print()
train_tf, val_tf = get_transforms(cfg)
ds  = EOSARDataset(cfg["data"]["root_dir"], "train", cfg["data"]["image_size"], train_tf)
img, mask, fname = ds[0]

print(f"Train samples  : {len(ds)}")
print(f"Image tensor   : {list(img.shape)}  dtype={img.dtype}  range=[{img.min():.3f}, {img.max():.3f}]")
print(f"Mask tensor    : {list(mask.shape)}  unique={mask.unique().tolist()}")
print(f"Sample file    : {fname}")

# ── Model forward pass ────────────────────────────────────────────────────────
print()
model = build_model(cfg)
model.eval()
dummy = torch.zeros(1, cfg["model"]["in_channels"],
                    cfg["data"]["image_size"], cfg["data"]["image_size"])
with torch.no_grad():
    out = model(dummy)
print(f"Forward pass   : {list(dummy.shape)} → {list(out.shape)}  ✓")

---
## § 5 — Visualise a Sample Tile

Displays the EO image, SAR image, and ground-truth mask for one training tile.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# img is already loaded from the sanity check cell above
eo   = img[:3].permute(1, 2, 0).numpy()   # (H, W, 3)
sar  = img[3].numpy()                      # (H, W)
gt   = mask[0].numpy()                     # (H, W)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f"Sample: {fname}", fontsize=11, y=1.01)

axes[0].imshow(eo)
axes[0].set_title("Pre-event — EO (RGB)", fontweight="bold")

axes[1].imshow(sar, cmap="gray")
axes[1].set_title("Post-event — SAR (grayscale)", fontweight="bold")

axes[2].imshow(gt, cmap="binary_r", vmin=0, vmax=1)
axes[2].set_title(f"Target mask  (change px: {int(gt.sum()):,})", fontweight="bold")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.savefig("sample_preview.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: sample_preview.png")

---
## § 6 — Training

### 6a — Debug Run (2 epochs, workers=0)
Run this first to confirm the full pipeline executes without errors on GPU.

In [ ]:
!python train.py --config configs/config.yaml --debug --name debug_run

### 6b — Full Training (40 epochs)

Expected time on T4 at 512×512 / batch 8:  ~3–5 min/epoch → **~2–3 hours total**.

Checkpoints auto-save to `experiments/kaggle_run_v1/checkpoints/` after every epoch (`latest.pth`) and on improvement (`best.pth`).

In [ ]:
!python train.py --config configs/config.yaml --name kaggle_run_v1

### 6c — Resume (if session timed out or was interrupted)

In [ ]:
import glob

checkpoints = sorted(glob.glob("experiments/kaggle_run_v1/checkpoints/latest.pth"))
if checkpoints:
    ckpt = checkpoints[-1]
    print(f"Resuming from: {ckpt}")
    !python train.py --config configs/config.yaml \
                     --name   kaggle_run_v1 \
                     --resume "{ckpt}"
else:
    print("No checkpoint found. Run §6b first.")

---
## § 7 — Evaluation

### 7a — Validation Set
Use during hyperparameter tuning. Run as many times as needed.

In [ ]:
import glob

best = "experiments/kaggle_run_v1/checkpoints/best.pth"
if os.path.exists(best):
    !python eval.py --config configs/config.yaml \
                    --checkpoint "{best}" \
                    --split val \
                    --n-vis 8
else:
    print("No best.pth found. Complete training first.")

### 7b — Test Set  ⚠️ Run Once Only

> **Only run this cell after all hyperparameter decisions are final.**  
> The test set covers scenes 09–10 (unseen geographic regions).  
> Running it repeatedly to guide model selection = data leakage.

In [ ]:
best = "experiments/kaggle_run_v1/checkpoints/best.pth"
if os.path.exists(best):
    !python eval.py --config configs/config.yaml \
                    --checkpoint "{best}" \
                    --split test \
                    --n-vis 16
else:
    print("No best.pth found.")

---
## § 8 — Results

### 8a — Metrics Summary

In [ ]:
import json, glob

for split in ("val", "test"):
    paths = sorted(glob.glob(f"experiments/*/eval_{split}/metrics.json"))
    if not paths:
        continue
    with open(paths[-1]) as f:
        r = json.load(f)
    print(f"\n{'═'*44}")
    print(f"  {split.upper()} SET  |  epoch {r.get('epoch', '?')}")
    print(f"{'═'*44}")
    print(f"  IoU        : {r['iou']:.4f}")
    print(f"  F1 Score   : {r['f1']:.4f}")
    print(f"  Precision  : {r['precision']:.4f}")
    print(f"  Recall     : {r['recall']:.4f}")
    print(f"  Loss       : {r['loss']:.4f}")
    print(f"{'═'*44}")

### 8b — Prediction Visualisations

In [ ]:
from IPython.display import Image as IPImage, display
import glob

for label, pattern in [
    ("Prediction grid (EO | SAR | GT | Pred)",
     "experiments/*/eval_test/predictions/predictions.png"),
    ("Error analysis  (green=TP, red=FP, blue=FN)",
     "experiments/*/eval_test/error_analysis/error_analysis.png"),
]:
    imgs = sorted(glob.glob(pattern))
    if imgs:
        print(f"\n{label}")
        display(IPImage(imgs[-1], width=900))
    else:
        print(f"[{label}] — not found (run §7b first)")

### 8c — Training Curves (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir experiments